<a href="https://colab.research.google.com/github/we-human-centric/CursoPythonDatos_2026/blob/main/Dia%2011%20-%202026-05-07%20-%20%C3%81rbol%20de%20Decisi%C3%B3n/clase_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Día 11 · Árbol de decisión

**Seminario EDA 2026 — Semana 3, Día 4 (jueves 7 de mayo)**

> Pasamos del modelo lineal logístico a un modelo **no lineal** que toma decisiones por preguntas anidadas (sí/no). Aprendemos:
>
> 1. Cómo el árbol parte el espacio de features con criterios de **Gini** o **entropía**.
> 2. Por qué un árbol sin restringir sobreajusta — y cómo controlar la profundidad.
> 3. **GridSearchCV** para tunear hyperparams sin trampa.
> 4. **Feature importance** — quién decide más en el árbol.

Acompaña la clase con: 🎮 [overfitting](../simuladores/simulador-overfitting.html)

## 0. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, StratifiedKFold
from sklearn.metrics import classification_report, roc_auc_score, accuracy_score, f1_score

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (8, 5)
print('Setup OK')

## 1. Mismo dataset que ayer (Pima Diabetes) — para comparar contra logística

Usar el mismo dataset es deliberado: el viernes vamos a sentar lado a lado las métricas de logística vs árbol y ver cuál gana.

In [ ]:
URL = 'https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv'
cols = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness',
        'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']
df = pd.read_csv(URL, names=cols)
X = df.drop(columns='Outcome')
y = df['Outcome'].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=SEED, stratify=y)
print('Train:', X_train.shape, '   Test:', X_test.shape)

## 2. Cálculo de Gini paso a paso

$G = 1 - \sum_k p_k^2$. Un nodo con 50/50 entre 2 clases tiene Gini = 0.5 (máxima impureza); un nodo puro (100% una clase) tiene Gini = 0.

Veamos un split sintético:

In [ ]:
def gini(p):
    """Gini para clasificación binaria con proporción p de clase 1."""
    return 1 - p**2 - (1-p)**2

padre_n, padre_p = 200, 0.50
hijo_izq_n, hijo_izq_p = 120, 0.30
hijo_der_n, hijo_der_p = 80,  0.80

G_padre = gini(padre_p)
G_izq   = gini(hijo_izq_p)
G_der   = gini(hijo_der_p)
G_pond  = (hijo_izq_n/padre_n) * G_izq + (hijo_der_n/padre_n) * G_der
ganancia = G_padre - G_pond

print(f'Gini padre  (n={padre_n}, p={padre_p}) = {G_padre:.4f}')
print(f'Gini izq    (n={hijo_izq_n}, p={hijo_izq_p}) = {G_izq:.4f}')
print(f'Gini der    (n={hijo_der_n}, p={hijo_der_p}) = {G_der:.4f}')
print(f'Gini ponderado de hijos             = {G_pond:.4f}')
print(f'Reducción de impureza (ganancia)    = {ganancia:.4f}')

El árbol de scikit-learn evalúa todas las posibles particiones (variable + umbral) y elige la que maximiza esta ganancia.

## 3. Árbol sin restringir — el camino al overfitting

Si dejamos crecer el árbol hasta hojas puras: 100% accuracy en train, mediocre en test.

In [ ]:
tree_max = DecisionTreeClassifier(random_state=SEED)
tree_max.fit(X_train, y_train)

print(f'Profundidad real      : {tree_max.get_depth()}')
print(f'# hojas               : {tree_max.get_n_leaves()}')
print(f'Accuracy train        : {tree_max.score(X_train, y_train):.3f}')
print(f'Accuracy test         : {tree_max.score(X_test,  y_test):.3f}')
print(f'AUC test              : {roc_auc_score(y_test, tree_max.predict_proba(X_test)[:,1]):.3f}')

**Síntoma claro de overfitting:** train ≈ 1.0, test ~0.7. El árbol memorizó el train. Vamos a podarlo.

## 4. Curva de complejidad — accuracy_train y accuracy_test vs max_depth

In [ ]:
depths = list(range(1, 21))
tr_scores, te_scores = [], []

for d in depths:
    m = DecisionTreeClassifier(max_depth=d, random_state=SEED).fit(X_train, y_train)
    tr_scores.append(m.score(X_train, y_train))
    te_scores.append(m.score(X_test,  y_test))

plt.plot(depths, tr_scores, marker='o', label='Accuracy TRAIN')
plt.plot(depths, te_scores, marker='s', label='Accuracy TEST')
plt.axvline(np.argmax(te_scores)+1, color='red', linestyle='--', alpha=0.5,
            label=f'Mejor depth en test = {np.argmax(te_scores)+1}')
plt.xlabel('max_depth'); plt.ylabel('Accuracy')
plt.title('Bias–Variance tradeoff en árboles')
plt.legend(); plt.tight_layout(); plt.show()

**Lectura:** train sube monotónicamente; test sube hasta cierta profundidad y luego se estanca o baja. Esa es la firma exacta del overfitting.

## 5. GridSearchCV — tunear hyperparams sin sesgar el test

GridSearchCV recorre combinaciones de hyperparams usando validación cruzada **dentro del train**. Solo al final evaluamos en test.

In [ ]:
param_grid = {
    'max_depth':         [3, 4, 5, 6, 8, 10, None],
    'min_samples_leaf':  [1, 5, 10, 20],
    'criterion':         ['gini', 'entropy'],
}
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

grid = GridSearchCV(DecisionTreeClassifier(random_state=SEED),
                    param_grid, cv=skf, scoring='roc_auc', n_jobs=-1)
grid.fit(X_train, y_train)

print('Mejores params :', grid.best_params_)
print(f'Mejor AUC (CV) : {grid.best_score_:.3f}')

best = grid.best_estimator_
y_prob_test = best.predict_proba(X_test)[:, 1]
y_pred_test = best.predict(X_test)
print(f'\nAUC en TEST    : {roc_auc_score(y_test, y_prob_test):.3f}')
print(f'Acc en TEST    : {accuracy_score(y_test, y_pred_test):.3f}')
print(f'F1 en TEST     : {f1_score(y_test, y_pred_test):.3f}')

## 6. Visualización del árbol podado

In [ ]:
fig, ax = plt.subplots(figsize=(20, 9))
plot_tree(best, feature_names=X.columns, class_names=['No diabetes', 'Diabetes'],
          filled=True, fontsize=9, rounded=True, ax=ax)
plt.title(f'Árbol de decisión podado (max_depth={best.get_depth()}, hojas={best.get_n_leaves()})')
plt.tight_layout(); plt.show()

**Cómo leer el árbol:**  
Cada nodo contiene la regla de decisión (`feature ≤ umbral`), Gini, número de muestras, distribución por clase y la clase predicha (más frecuente). Color cargado = más puro. La mayoría de árboles publicados "a la luz" se ven así.

## 7. Feature importance — quién decide más

In [ ]:
imp = pd.Series(best.feature_importances_, index=X.columns).sort_values(ascending=True)
imp.plot(kind='barh', figsize=(8, 5))
plt.title('Feature importance (árbol final)')
plt.xlabel('Importancia (suma reducción Gini ponderada)')
plt.tight_layout(); plt.show()
print(imp.sort_values(ascending=False).round(3))

Glucose suele dominar (tiene sentido clínico). Importante: la importancia del árbol es *cuánto reduce Gini en los splits*; **no** indica dirección (positiva/negativa) como sí hacen los coeficientes de la logística.

## 8. Ejercicio — comparar logística vs árbol con CV

Completa el dict `modelos` con los dos clasificadores y ejecuta. Luego comenta cuál es más estable (std bajo) y cuál tiene mejor AUC.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

# TODO: completa los placeholders ___
modelos = {
    'Logística': Pipeline([('s', StandardScaler()), ('m', LogisticRegression(max_iter=1000, random_state=SEED))]),
    'Árbol':     DecisionTreeClassifier(max_depth=best.get_depth(), random_state=SEED),  # ← podemos usar el mejor depth
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
for nombre, m in modelos.items():
    aucs = cross_val_score(m, X, y, cv=skf, scoring='roc_auc')
    print(f'{nombre:<10s}  AUC = {aucs.mean():.3f} ± {aucs.std():.3f}   (folds: {np.round(aucs, 3)})')

In [ ]:
# Asserts
assert len(modelos) == 2
print('✓ Ejercicio completado — escribe en una celda markdown debajo cuál ganó y por qué')

**Tu respuesta** (edita esta celda):

← Escribe aquí cuál de los dos modelos ganó en AUC media y cuál es más estable (menor std). 2-3 oraciones.

## 9. Cost-Complexity Pruning (alternativa a max_depth)

Otra forma de podar: penalizar cada hoja con un costo $\alpha$ y elegir el árbol que minimiza error + α·n_hojas.

In [ ]:
path = DecisionTreeClassifier(random_state=SEED).cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas[:-1]   # quitamos el último (árbol trivial)

trees = [DecisionTreeClassifier(random_state=SEED, ccp_alpha=a).fit(X_train, y_train) for a in ccp_alphas]
tr_acc = [t.score(X_train, y_train) for t in trees]
te_acc = [t.score(X_test,  y_test)  for t in trees]
n_leaves = [t.get_n_leaves() for t in trees]

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ccp_alphas, tr_acc, marker='o', label='train')
ax[0].plot(ccp_alphas, te_acc, marker='s', label='test')
ax[0].set_xlabel('ccp_alpha'); ax[0].set_ylabel('Accuracy')
ax[0].set_title('Accuracy vs ccp_alpha'); ax[0].legend()
ax[1].plot(ccp_alphas, n_leaves, marker='o')
ax[1].set_xlabel('ccp_alpha'); ax[1].set_ylabel('# hojas')
ax[1].set_title('Tamaño del árbol')
plt.tight_layout(); plt.show()

## 10. Tabla comparativa lineal vs logística vs árbol

| Aspecto | Lineal | Logística | Árbol |
|---|---|---|---|
| Tipo de problema | regresión | clasificación | regresión y clasificación |
| Frontera | hiperplano | hiperplano (en log-odds) | rectángulos en el espacio de features |
| Supuestos del OLS | ✓ requiere | ✗ no | ✗ no |
| No-linealidad | manual (polinomial) | manual | nativa |
| Interpretabilidad coefs | excelente | excelente (odds ratios) | regla a regla |
| Sensibilidad a outliers | alta | media | baja |
| Escala de features | indiferente al fit, pero importa con regularización | importa | indiferente |
| Feature importance | β_j (con dirección) | β_j (con dirección) | importance (sin dirección) |

## 11. Entregable parcial S3

Sobre tu dataset:

1. Ajusta un `DecisionTreeClassifier` (o `Regressor`) sobre la variable objetivo.  
2. Hyperparametros: `max_depth ∈ {3,5,7,10,None}`, `min_samples_leaf ∈ {1,5,10}`. Usa GridSearchCV con CV-5.  
3. Reporta los mejores hyperparams y el AUC (o R²) en el test.  
4. Visualiza el árbol final con `plot_tree`.  
5. Reporta los 3 features más importantes y comenta si tiene sentido de negocio.

Mañana cerramos la semana con métricas detalladas y la entrega oficial S3.